# 计算嵌入向量的距离矩阵统计量 (使用 pdist)

### 目的
本脚本加载多个 Stage 2 S蛋白嵌入向量数据集（`.pkl`文件），对每个数据集中的嵌入向量使用`scipy.spatial.distance.pdist`计算多种距离度量的压缩距离矩阵。然后，计算每个压缩矩阵的均值和标准差，以量化不同嵌入空间的整体分散性。

### 流程
1.  **参数配置**：设置输入目录、输出CSV路径以及要计算的距离度量列表 (`METRICS`)。
2.  **查找文件**：自动查找输入目录下的所有 `S_val_emb_*.pkl` 文件。
3.  **主处理循环**：
    a. 遍历每个PKL文件。
    b. **加载数据**并提取嵌入向量。
    c. **数据标准化**：对嵌入向量进行标准化（可选，但通常推荐）。
    d. **循环计算距离**：对`METRICS`列表中的每种距离度量：
        i. 使用 `pdist` 计算压缩距离矩阵。
        ii. 计算压缩矩阵的均值和标准差。
        iii. 记录文件名、度量名称、均值和标准差。
4.  **保存结果**：将所有结果汇总到DataFrame并保存为CSV。

### 1. 导入所需库

In [5]:
import os
import pickle
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform # <--- 导入 pdist
from sklearn.preprocessing import StandardScaler # <--- 导入 StandardScaler
from tqdm.notebook import tqdm
import glob

print("所有库导入成功。")

所有库导入成功。


### 2. 参数配置

In [6]:
# ----- 1. 输入目录 -----
INPUT_DIR = "/data2/zhoukaitao/01evoModel/dataset/251026_S_Val_Emb/"

# ----- 2. 输出配置 -----
OUTPUT_DIR = "/data2/zhoukaitao/01evoModel/result/review/8similarity/"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "embedding_distance_stats_pdist.csv")

# ----- 3. 数据配置 -----
PROTEIN_KEY = "S"

# ----- 4. 距离度量配置 -----
# 选择 pdist 支持的距离度量
# 常用: 'euclidean', 'cosine', 'cityblock' (曼哈顿距离), 'seuclidean', 'mahalanobis'...
# 查阅 scipy.spatial.distance.pdist 文档获取完整列表
METRICS = ['euclidean', 'cosine', 'cityblock']

# ----- 5. 是否标准化数据 -----
# 对于欧氏距离等受尺度影响的度量，通常推荐标准化
STANDARDIZE_DATA = True 

# 自动创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"输入目录: {INPUT_DIR}")
print(f"输出CSV文件: {OUTPUT_CSV}")
print(f"将计算的距离度量: {METRICS}")
print(f"是否标准化数据: {STANDARDIZE_DATA}")

输入目录: /data2/zhoukaitao/01evoModel/dataset/251026_S_Val_Emb/
输出CSV文件: /data2/zhoukaitao/01evoModel/result/review/8similarity/embedding_distance_stats_pdist.csv
将计算的距离度量: ['euclidean', 'cosine', 'cityblock']
是否标准化数据: True


### 3. 主处理循环
【修改】使用 pdist 计算距离。

In [7]:
# 查找输入目录下的所有 .pkl 文件
pkl_files = glob.glob(os.path.join(INPUT_DIR, "S_val_emb_ESM3_Struct_7country.pkl"))

results_list = []

if not pkl_files:
    print(f"错误：在目录 {INPUT_DIR} 中未找到任何 'S_val_emb_*.pkl' 文件。")
else:
    print(f"找到 {len(pkl_files)} 个PKL文件，开始处理...")

    for pkl_file_path in tqdm(pkl_files, desc="处理PKL文件"):
        filename = os.path.basename(pkl_file_path)
        print(f"\n--- 开始处理: {filename} ---")

        try:
            # 1. 加载数据并提取嵌入向量
            print("  加载数据...")
            with open(pkl_file_path, 'rb') as f:
                data = pickle.load(f)
            
            embeddings_list = []
            for sample in data:
                if PROTEIN_KEY in sample:
                    embeddings_list.append(sample[PROTEIN_KEY])
                else:
                    print(f"    警告: 文件 {filename} 中样本 {sample.get('id', 'Unknown')} 缺少键 '{PROTEIN_KEY}'，已跳过。")
            
            if not embeddings_list:
                print("    错误：未能从文件中提取任何有效的嵌入向量。跳过此文件。")
                continue
            
            X = np.array(embeddings_list)
            if X.ndim == 1: X = X.reshape(-1, 1)
            elif X.ndim > 2: X = X.reshape(X.shape[0], -1)
            print(f"    数据维度: {X.shape}")
            
            if X.shape[0] < 2:
                print("    错误：样本数不足2个，无法计算距离。跳过此文件。")
                continue
                
            # 2. 【可选】标准化数据
            if STANDARDIZE_DATA:
                print("  标准化数据...")
                scaler = StandardScaler()
                X_proc = scaler.fit_transform(X) # 使用处理后的数据进行计算
            else:
                X_proc = X # 直接使用原始数据
                
            # 3. 【修改】循环计算多种距离度量
            print("  开始计算距离矩阵...")
            for metric in METRICS:
                print(f"    -- 计算 {metric} 距离 --")
                try:
                    # 使用 pdist 计算压缩距离矩阵
                    # 注意：pdist 计算的是距离，对于 'cosine'，它计算的是 1 - cosine_similarity
                    condensed_dist_matrix = pdist(X_proc, metric=metric)
                    
                    # 计算均值和标准差
                    mean_dist = np.mean(condensed_dist_matrix)
                    std_dist = np.std(condensed_dist_matrix)
                    
                    results_list.append({
                        'filename': filename,
                        'metric': metric,
                        'standardized': STANDARDIZE_DATA,
                        'mean': mean_dist,
                        'std_dev': std_dist
                    })
                    
                    print(f"      压缩矩阵形状: {condensed_dist_matrix.shape}")
                    print(f"      均值: {mean_dist:.6f}")
                    print(f"      标准差: {std_dist:.6f}")

                except ValueError as ve:
                    print(f"      计算 {metric} 距离时出错: {ve}")
                    print("      可能是因为数据包含 NaN/Inf 或度量不适用。")
                    results_list.append({
                        'filename': filename, 'metric': metric, 'standardized': STANDARDIZE_DATA,
                        'mean': np.nan, 'std_dev': np.nan, 'error': str(ve)
                    })
                except Exception as e:
                    print(f"      计算 {metric} 距离时发生未知错误: {e}")
                    results_list.append({
                        'filename': filename, 'metric': metric, 'standardized': STANDARDIZE_DATA,
                        'mean': np.nan, 'std_dev': np.nan, 'error': str(e)
                    })

            print(f"--- 完成处理: {filename} ---")
            
        except Exception as e:
            print(f"处理文件 {filename} 时发生意外错误: {e}")
            # 记录文件级别的错误
            results_list.append({'filename': filename, 'metric': 'File Error', 'standardized': STANDARDIZE_DATA,
                                 'mean': np.nan, 'std_dev': np.nan, 'error': str(e)})
            continue

print("\n=== 所有文件处理完毕 ===")

找到 1 个PKL文件，开始处理...


处理PKL文件:   0%|          | 0/1 [00:00<?, ?it/s]


--- 开始处理: S_val_emb_ESM3_Struct_7country.pkl ---
  加载数据...
    数据维度: (7000, 1536)
  标准化数据...
  开始计算距离矩阵...
    -- 计算 euclidean 距离 --
      压缩矩阵形状: (24496500,)
      均值: 45.466017
      标准差: 31.706154
    -- 计算 cosine 距离 --
      压缩矩阵形状: (24496500,)
      均值: 0.908420
      标准差: 0.636293
    -- 计算 cityblock 距离 --
      压缩矩阵形状: (24496500,)
      均值: 1511.080168
      标准差: 1078.346948
--- 完成处理: S_val_emb_ESM3_Struct_7country.pkl ---

=== 所有文件处理完毕 ===


### 4. 保存结果到CSV

In [8]:
if results_list:
    df_results = pd.DataFrame(results_list)
    print("\n--- 最终结果汇总 ---")
    display(df_results)
    
    try:
        df_results.to_csv(OUTPUT_CSV, index=False)
        print(f"\n结果已成功保存至: {OUTPUT_CSV}")
    except Exception as e:
        print(f"\n保存CSV文件时出错: {e}")
else:
    print("\n没有成功处理任何文件或计算任何距离，未生成结果CSV。")


--- 最终结果汇总 ---


,filename,metric,standardized,mean,std_dev
0,S_val_emb_ESM3_Struct_7country.pkl,euclidean,True,45.466017,31.706154
1,S_val_emb_ESM3_Struct_7country.pkl,cosine,True,0.908420,0.636293
2,S_val_emb_ESM3_Struct_7country.pkl,cityblock,True,1511.080168,1078.346948



结果已成功保存至: /data2/zhoukaitao/01evoModel/result/review/8similarity/embedding_distance_stats_pdist.csv
